# 語音 LoRA：Colab 練習

先閱讀第一課並取得錄音授權。切換至 GPU 執行階段；此筆記本會下載模型與套件，並讀寫私人 Google Drive。請逐格執行與確認輸出。

In [2]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [3]:
import torch
assert torch.cuda.is_available(), "請在執行階段設定選擇 GPU"
print("GPU:", torch.cuda.get_device_name(0))


GPU: Tesla T4


## 1. 取得教材與依賴

環境版本可能隨 Colab 更新而變；安裝後如提示重新啟動，請重新執行前面儲存格。

In [4]:
!git clone https://github.com/RichTsai1688/tts-lora-learning.git /content/tts-lora-learning
%cd /content/tts-lora-learning
%pip install -q qwen-tts==0.1.1 peft==0.18.1 tensorboard


Cloning into '/content/tts-lora-learning'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 21 (delta 0), reused 21 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 25.26 KiB | 5.05 MiB/s, done.
/content/tts-lora-learning
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [11]:
# 輸入你想合成的大富翁台詞
text = "玩具城市大富翁，準備擲骰子！"
print("準備合成語音台詞：", text)

準備合成語音台詞： 玩具城市大富翁，準備擲骰子！


In [16]:
# 1. 安裝線上繁體中文語音合成套件
!pip install gTTS

from gtts import gTTS
import IPython.display as ipd

# 2. 設定你的大富翁展示台詞
text = "玩具城市大富翁，偵測到 M5Stack 搖動，準備擲骰子！"
print("正在合成繁體中文語音台詞：", text)

# 3. 合成語音檔 (zh-TW 繁體中文)
tts = gTTS(text=text, lang='zh-TW')
output_filename = "shake_dice.mp3"
tts.save(output_filename)

print("✅ 語音檔生成成功！")

# 4. 在網頁中顯示播放列（線上試聽與下載）
ipd.Audio(output_filename)

  Attempting uninstall: click
    Found existing installation: click 8.5.0
    Uninstalling click-8.5.0:
      Successfully uninstalled click-8.5.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gTTS]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
spacy 3.8.18 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.


正在合成繁體中文語音台詞： 玩具城市大富翁，偵測到 M5Stack 搖動，準備擲骰子！
✅ 語音檔生成成功！


## 2. 設定私有資料與實驗名稱

CSV 中的音訊路徑以 CSV 所在資料夾為基準；參考錄音須為另一段同說話者錄音。每次重跑請更換 RUN_NAME。

## 3. 盤點與分組

閱讀報表，人工檢查授權、音訊／文字吻合、各組語言與語調分布；不符合時修正資料並更換實驗名稱。

In [6]:
subprocess.run([sys.executable, "scripts/dataset_tool.py", str(MANIFEST), "--ref-audio", str(REFERENCE), "--output", str(RUN / "raw"), "--seed", "42"], check=True)


CalledProcessError: Command '['/usr/bin/python3', 'scripts/dataset_tool.py', '/content/drive/MyDrive/tts-private/manifest.csv', '--ref-audio', '/content/drive/MyDrive/tts-private/reference.wav', '--output', '/content/drive/MyDrive/tts-private/runs/lesson-smoke-01/raw', '--seed', '42']' returned non-zero exit status 1.

## 4. 將錄音轉為模型訓練格式

會下載 tokenizer，依資料量花費時間。

In [7]:
encoded = RUN / "encoded"
encoded.mkdir()
for split in ("train", "val", "test"):
    subprocess.run([sys.executable, "training/prepare_data.py", "--device", "cuda:0", "--input_jsonl", str(RUN / "raw" / f"{split}_raw.jsonl"), "--output_jsonl", str(encoded / f"{split}.jsonl")], check=True)


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/tts-private/runs/lesson-smoke-01/encoded'

## 5. 訓練與檢查

先做 smoke test；正式訓練請重新設定實驗名稱與 SMOKE。下載基礎模型與輸出合併模型需要足夠空間；fp16 模式以 float32 載入基礎權重，可能需要較多 GPU 記憶體。

In [8]:
cmd = [sys.executable, "training/train_lora.py", "--init_model_path", "Qwen/Qwen3-TTS-12Hz-0.6B-Base", "--dataset_dir", str(encoded), "--output_model_path", str(RUN / "model"), "--speaker_name", "speaker_demo", "--batch_size", "1", "--num_epochs", "1" if SMOKE else "3", "--precision", PRECISION]
if SMOKE:
    cmd += ["--max_train_batches", "4", "--max_eval_batches", "1"]
subprocess.run(cmd, check=True)
print("私人輸出位置:", RUN / "model")


CalledProcessError: Command '['/usr/bin/python3', 'training/train_lora.py', '--init_model_path', 'Qwen/Qwen3-TTS-12Hz-0.6B-Base', '--dataset_dir', '/content/drive/MyDrive/tts-private/runs/lesson-smoke-01/encoded', '--output_model_path', '/content/drive/MyDrive/tts-private/runs/lesson-smoke-01/model', '--speaker_name', 'speaker_demo', '--batch_size', '1', '--num_epochs', '1', '--precision', 'fp16', '--max_train_batches', '4', '--max_eval_batches', '1']' returned non-zero exit status 1.

檢視 `loss_history.csv` 與 `final_model/`。這是合併後的完整模型，請用未參與訓練的固定測試句人工聽測並記錄限制。